# Preprocessing the RNA degradation data
1. Standardize the dataset -> huggingface dataset
2. embed to the pipeline
3. run the model
4. output the results

- Degradation dataset (ok)
- Secondary structure dataset (doing)
- 

In [5]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split

In [73]:
#loading the training dataset

def to_df(tag = "train", json_file = ""):
    private_df = pd.read_csv("/home/sxr280/BERTLocRNA/data/RNA_degradation/private_test_labels.csv")
    data = []
    prefix = "/home/sxr280/BERTLocRNA/data/RNA_degradation/"
    with open(prefix + json_file, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    df = pd.DataFrame(data)
    df = df.drop(columns=["index"])
    # print(df.columns, df["signal_to_noise"])
    keep_col = set(df.columns).intersection(set(private_df.columns))
    df = df.loc[:, list(keep_col)]
    private_df = private_df.loc[:, list(keep_col)]
    # print(df["signal_to_noise"])
    # df["SN_filter"] = df["SN_filter"].astype("int")

    train_idx, val_idx = train_test_split(range(df.shape[0]), test_size=0.2, random_state=42)
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    train_df.to_csv("/home/sxr280/BERTLocRNA/data/RNA_degradation/Train.csv", sep = ",", index=False)
    val_df.to_csv("/home/sxr280/BERTLocRNA/data/RNA_degradation/Val.csv", sep = ",", index=False)
    private_df.to_csv("/home/sxr280/BERTLocRNA/data/RNA_degradation/Test.csv", sep = ",", index=False)
    return train_df, val_df, private_df

train_df, val_df, test_df = to_df(json_file = "train.json")

# Checking the RNA degradation dataset

In [15]:
from datasets import load_dataset, Value, Features
data = load_dataset("TerminatorJ/RNA_degradation", split = "validation")#, features=custom_features)

: 

# Checking the AchiveII dataset

In [22]:
import pickle
import _pickle as cPickle
import sys
import os
import torch.optim as optim
from torch.utils import data

from e2efold.models import ContactNetwork, ContactNetwork_test, ContactNetwork_fc
from e2efold.models import ContactAttention, ContactAttention_simple_fix_PE
from e2efold.models import ContactAttention_simple
from e2efold.common.utils import *
from e2efold.common.config import process_config
from e2efold.postprocess import postprocess
from e2efold.data_generator import RNASSDataGenerator, Dataset
import collections

RNA_SS_data = collections.namedtuple('RNA_SS_data', 'seq ss_label length name pairs')
train_data = RNASSDataGenerator("/home/sxr280/BERTLocRNA/data/RNA_secondary_structure/e2efold-master/data/archiveII_all/", "train")
test_data = RNASSDataGenerator("/home/sxr280/BERTLocRNA/data/RNA_secondary_structure/e2efold-master/data/archiveII_all/", "test")
val_data = RNASSDataGenerator("/home/sxr280/BERTLocRNA/data/RNA_secondary_structure/e2efold-master/data/archiveII_all/", "val")




In [46]:
#decode
label_dict = {
    '.': np.array([1,0,0]), 
    '(': np.array([0,1,0]), 
    ')': np.array([0,0,1])
}
seq_dict = {
    'A':np.array([1,0,0,0]),
    'U':np.array([0,1,0,0]),
    'C':np.array([0,0,1,0]),
    'G':np.array([0,0,0,1]),
    'N':np.array([0,0,0,0])
}

def decode_ss(encode):
    ss_label = ["N"]*len(encode)
    # print(ss_label)
    for i in range(len(encode)):
        if encode[i][0] == 1:
            # print(ss_label[i])
            ss_label[i] = "."
        elif encode[i][1] == 1:
            ss_label[i] = "("
        elif encode[i][2] == 1:
            ss_label[i] = ")"
    # assert "N" not in ss_label, "somewhere not inserted"
    return "".join(ss_label)

def decode_seq(encode):
    seq_label = ["N"]*len(encode)
    # print(ss_label)
    for i in range(len(encode)):
        if encode[i][0] == 1:
            # print(ss_label[i])
            seq_label[i] = "A"
        elif encode[i][1] == 1:
            seq_label[i] = "U"
        elif encode[i][2] == 1:
            seq_label[i] = "C"
        elif encode[i][3] == 1:
            seq_label[i] = "G"
        else:
            seq_label[i] = "N"
    # assert "N" not in ss_label, "somewhere not inserted"
    return "".join(seq_label)


train_label = list(map(lambda x: decode_ss(x), train_data.data_y))
train_seq = list(map(lambda x: decode_seq(x), train_data.data_x))
train_seq_len = list(map(lambda x: x, train_data.seq_length))

test_label = list(map(lambda x: decode_ss(x), test_data.data_y))
test_seq = list(map(lambda x: decode_seq(x), test_data.data_x))
test_seq_len = list(map(lambda x: x, test_data.seq_length))

val_label = list(map(lambda x: decode_ss(x), val_data.data_y))
val_seq = list(map(lambda x: decode_seq(x), val_data.data_x))
val_seq_len = list(map(lambda x: x, val_data.seq_length))

In [ ]:
custom_features = Features({"label": Value(dtype='string'), 
                                "idx": Value(dtype="int64"),
                                "Xall": Value(dtype='string'),
                                "Xtag": Value(dtype="int64"),
                                "ids": Value(dtype='string')})

In [50]:
#save to the dataframe
train_df = pd.DataFrame({"Seq": train_seq, "label": train_label, "length": train_seq_len})
test_df = pd.DataFrame({"Seq": test_seq, "label": test_label, "length": test_seq_len})
val_df = pd.DataFrame({"Seq": val_seq, "label": val_label, "length": val_seq_len})

In [53]:
#save to csv file
train_df.to_csv("/home/sxr280/BERTLocRNA/data/RNA_secondary_structure/e2efold-master/data/Train.csv", sep = ",", index=False)
test_df.to_csv("/home/sxr280/BERTLocRNA/data/RNA_secondary_structure/e2efold-master/data/Test.csv", sep = ",", index=False)
val_df.to_csv("/home/sxr280/BERTLocRNA/data/RNA_secondary_structure/e2efold-master/data/Val.csv", sep = ",", index=False)

# Test to load the RNAstralign dataset from remote directory

In [11]:
from datasets import load_dataset, Value, Features
data = load_dataset("TerminatorJ/RNAstralign", split = "validation")